In [6]:
from transformer_lens.HookedTransformer import HookedTransformer
from transformers import AutoTokenizer
from dataset import IOIDataset
import torch
import json

def get_pythia_70m(device="cuda") -> HookedTransformer:
    tl_model = HookedTransformer.from_pretrained("EleutherAI/pythia-70m-deduped")
    tl_model = tl_model.to(device)
    tl_model.set_use_attn_result(True)
    tl_model.set_use_split_qkv_input(True)
    if "use_hook_mlp_in" in tl_model.cfg.to_dict():
        tl_model.set_use_hook_mlp_in(True)
    return tl_model


In [2]:
num_examples = 100
device = "cuda:0"

tl_model = get_pythia_70m(device=device)
ioi_dataset = IOIDataset(
    prompt_type="ABBA",
    N=num_examples*2,
    nb_templates=1,
    seed = 0,
)

tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-70m-deduped")

Loaded pretrained model EleutherAI/pythia-70m-deduped into HookedTransformer
Moving model to device:  cuda:0


In [7]:
clean_ds = ioi_dataset.ioi_prompts
len_prefix = len(tokenizer.encode(clean_ds[0]['text'][:-(len(clean_ds[0]['IO']) + 1)]))
len_sol = 1
all_data = []

for prompt_idx in range(len(ioi_dataset.ioi_prompts)):
    
    text = clean_ds[prompt_idx]['text']
    clean_sol = " " + clean_ds[prompt_idx]['IO']
    corrupted_sol = " " + clean_ds[prompt_idx]['S']
    corrupted_text = text.replace(clean_sol, "[IO]").replace(corrupted_sol, "[S]")
    corrupted_text = corrupted_text.replace("[IO]", corrupted_sol).replace("[S]", clean_sol)
    assert text.endswith(clean_sol), f"Solution {clean_sol} not at end of text {text}"
    assert corrupted_text.endswith(corrupted_sol), f"Solution {corrupted_sol} not at end of text {corrupted_text}"
    clean_text = text[:-len(clean_sol)]
    corrupted_text = corrupted_text[:-len(corrupted_sol)]
    assert len(tokenizer.encode(clean_text)) == len_prefix, f"Length of clean text {len(clean_text)} not equal to length of prefix {len_prefix}"
    assert len(tokenizer.encode(corrupted_text)) == len_prefix, f"Length of corrupted text {len(corrupted_text)} not equal to length of prefix {len_prefix}"
    assert len(tokenizer.encode(clean_sol)) == len_sol, f"length of clean_sol {len(clean_sol)} not equal to {len_sol}"
    assert len(tokenizer.encode(corrupted_sol)) == len_sol, f"length of corrupted_sol {len(corrupted_sol)} not equal to {len_sol}"
    print("clean_text: ", clean_text)
    print("corrupted_text: ", corrupted_text)
    print("\n")
    
    # Add the current data to the list
    data = {
        "clean_prefix": clean_text,
        "patch_prefix": corrupted_text,
        "clean_answer": clean_sol,
        "patch_answer": corrupted_sol,
        "case": "ioi_dataset"
    }
    all_data.append(data)

with open('ioi_examples.json', 'w') as f:
    for item in all_data:
        json.dump(item, f)
        f.write('\n')    
    
    

clean_text:  Then, Alexander and Michelle went to the restaurant. Michelle gave a bone to
corrupted_text:  Then, Michelle and Alexander went to the restaurant. Alexander gave a bone to


clean_text:  Then, Richard and Paul went to the garden. Paul gave a necklace to
corrupted_text:  Then, Paul and Richard went to the garden. Richard gave a necklace to


clean_text:  Then, Finn and Jeremy went to the station. Jeremy gave a drink to
corrupted_text:  Then, Jeremy and Finn went to the station. Finn gave a drink to


clean_text:  Then, Jonathan and Adam went to the garden. Adam gave a bone to
corrupted_text:  Then, Adam and Jonathan went to the garden. Jonathan gave a bone to


clean_text:  Then, Helen and Benjamin went to the garden. Benjamin gave a computer to
corrupted_text:  Then, Benjamin and Helen went to the garden. Helen gave a computer to


clean_text:  Then, Bradley and Eric went to the house. Eric gave a kiss to
corrupted_text:  Then, Eric and Bradley went to the house. Bradley g